# Week 5 — Lab
## Attribution on tabular, text, and image inputs

Three end-to-end attribution workflows:

1. **SHAP TreeExplainer** on a gradient-boosting regressor for California housing.
2. **LIME** on a TF-IDF + Logistic Regression text classifier for 20-newsgroups.
3. **Captum Integrated Gradients** on a small PyTorch CNN trained on FashionMNIST.

Each section answers two questions: *what does the model think is important here?* and
*should we believe it?*


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.style.use("../../assets/mplstyle/course.mplstyle")
RNG = np.random.default_rng(0)


## Part A — SHAP on a tabular regressor

We train a `GradientBoostingRegressor` on California housing and use
`shap.TreeExplainer` to compute exact Shapley values.

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
import shap

data = fetch_california_housing(as_frame=True)
X, y = data.data, data.target
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0)

reg = GradientBoostingRegressor(random_state=0, n_estimators=300, max_depth=4).fit(Xtr, ytr)
print(f"Test R²: {reg.score(Xte, yte):.3f}")

explainer = shap.TreeExplainer(reg)
sv = explainer(Xte.iloc[:2000])      # SHAP Explanation object, shape (n, n_features)
print(f"SHAP values shape: {sv.values.shape}")
print(f"Expected value (baseline f̄): {sv.base_values[0]:.3f}")


### A1 — Global summary (beeswarm)

The summary plot shows, for each feature, the **distribution of its SHAP values**
across the test set, coloured by the feature value. It is the single most informative
SHAP chart.

In [ ]:
shap.plots.beeswarm(sv, max_display=10, show=False)
plt.title("SHAP summary — California housing")
plt.tight_layout(); plt.show()


**Reading.** For `MedInc` (median income) the SHAP values fan out wide and are
strongly correlated with the feature value (red points on the right) — high income
pushes the prediction up. `Latitude` and `Longitude` show a roughly symmetric spread
around zero — being in a "good" location adds to the prediction, a "bad" one subtracts.
`HouseAge`, by contrast, sits close to zero — the model barely uses it.

### A2 — Global feature importance (mean |SHAP|)

In [ ]:
shap.plots.bar(sv, max_display=10, show=False)
plt.title("Mean |SHAP| — California housing")
plt.tight_layout(); plt.show()


This is the SHAP-equivalent of the gini importance bar from week 3 — but unbiased
toward high-cardinality features, because Shapley axioms guarantee it. Use this as your
default global-importance chart for tree models.

### A3 — Local explanation for a single prediction

Pick one test point and explain its prediction as a budget: baseline + per-feature
contributions sum exactly to $f(x)$.

In [ ]:
# Pick a prediction that is far from the mean, to make the plot informative
i = int(np.argmax(reg.predict(Xte)[:2000]))
print(f"True: {yte.iloc[i]:.3f}    Predicted: {reg.predict(Xte.iloc[[i]])[0]:.3f}")

shap.plots.waterfall(sv[i], max_display=10, show=False)
plt.title(f"Local SHAP for test row {i}")
plt.tight_layout(); plt.show()


**Reading.** The bottom of the waterfall is $E[f(X)] \approx 2.07$ (the dataset
mean). Each red bar pushes the prediction up by its width; each blue bar pushes it
down. The top of the waterfall is the prediction $f(x)$. The numeric labels on each bar
are the SHAP value of that feature for **this specific row**.

This is the chart you put next to a single prediction in a paper to say "the model
predicted this because…".

## Part B — LIME on a text classifier

We train a TF-IDF + LogReg classifier on a 4-class subset of 20-newsgroups, then ask
LIME to explain a single document at the token level.

In [ ]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from lime.lime_text import LimeTextExplainer

cats = ["sci.space", "sci.med", "rec.sport.hockey", "comp.graphics"]
train = fetch_20newsgroups(subset="train", categories=cats,
                           remove=("headers", "footers", "quotes"))
test  = fetch_20newsgroups(subset="test", categories=cats,
                           remove=("headers", "footers", "quotes"))

pipe = make_pipeline(
    TfidfVectorizer(max_features=20000, min_df=3, ngram_range=(1, 2)),
    LogisticRegression(max_iter=2000, C=1.0),
)
pipe.fit(train.data, train.target)
print(f"Test accuracy: {pipe.score(test.data, test.target):.3f}")


In [ ]:
# A correctly-classified test document
predictions = pipe.predict(test.data)
ok = np.where(predictions == np.array(test.target))[0]
idx = ok[3]
print(f"True class:      {cats[test.target[idx]]}")
print(f"Predicted class: {cats[predictions[idx]]}")
print(f"---\n{test.data[idx][:600]}...\n---")


In [ ]:
explainer = LimeTextExplainer(class_names=cats)
exp = explainer.explain_instance(
    test.data[idx],
    pipe.predict_proba,
    num_features=10,
    num_samples=2000,
    labels=[test.target[idx]],
)

# Pull the LIME explanation as a list of (token, weight) and plot
tokens, weights = zip(*exp.as_list(label=test.target[idx]))
order = np.argsort(weights)
tokens = [tokens[i] for i in order]
weights = [weights[i] for i in order]

fig, ax = plt.subplots(figsize=(7, 5))
colors = ["#D55E00" if w < 0 else "#0072B2" for w in weights]
ax.barh(range(len(tokens)), weights, color=colors)
ax.set_yticks(range(len(tokens)), tokens)
ax.axvline(0, color="black", lw=0.7)
ax.set_xlabel(f"LIME weight (class = {cats[test.target[idx]]})")
ax.set_title("LIME token attribution")
plt.tight_layout(); plt.show()


**Reading.** Blue tokens push the prediction *toward* the target class; orange
tokens push *away*. Notice that the strongest tokens are usually domain-specific
content words, not stopwords — a sanity check that the classifier is doing something
reasonable.

### LIME instability check

LIME results depend on the random perturbation sample. Run it twice with different
seeds and see how much the top-10 differs.

In [ ]:
def lime_top_words(seed, k=10):
    explainer = LimeTextExplainer(class_names=cats, random_state=seed)
    exp = explainer.explain_instance(
        test.data[idx], pipe.predict_proba, num_features=k, num_samples=2000,
        labels=[test.target[idx]],
    )
    return [t for t, _ in exp.as_list(label=test.target[idx])]


a = set(lime_top_words(0))
b = set(lime_top_words(1))
print(f"Run 1 top-10: {sorted(a)}")
print(f"Run 2 top-10: {sorted(b)}")
print(f"Intersection: {len(a & b)} / 10")


**Reading.** On a few thousand perturbation samples the top-10 typically agrees
on 7–9 tokens — usable but not deterministic. If you cannot tolerate that variance,
raise `num_samples` or switch to SHAP.

## Part C — Captum Integrated Gradients on a CNN

Same setup as week 4: a tiny CNN trained briefly on FashionMNIST. We then run
**Integrated Gradients** on a few correctly-classified test images and visualize the
attribution as a heatmap.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import torchvision
from torchvision import transforms as T
from captum.attr import IntegratedGradients

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)

class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(64 * 7 * 7, 128), nn.ReLU(),
            nn.Linear(128, 10),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

tfm = T.Compose([T.ToTensor(), T.Normalize((0.286,), (0.353,))])
train = torchvision.datasets.FashionMNIST("./data", train=True, download=True, transform=tfm)
test  = torchvision.datasets.FashionMNIST("./data", train=False, download=True, transform=tfm)
classes = train.classes
tr_loader = DataLoader(Subset(train, list(range(6000))), batch_size=128, shuffle=True)
te_loader = DataLoader(Subset(test,  list(range(1000))), batch_size=128)

model = SmallCNN().to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)

for ep in range(3):
    model.train()
    for x, y in tr_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad(); loss = F.cross_entropy(model(x), y)
        loss.backward(); opt.step()

# Eval
model.eval()
correct, n = 0, 0
with torch.no_grad():
    for x, y in te_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        correct += (model(x).argmax(1) == y).sum().item(); n += x.size(0)
print(f"Test accuracy: {correct / n:.3f}")


In [ ]:
# Run IG on a handful of test images
ig = IntegratedGradients(model)
# Baseline: a constant black-image-equivalent in the normalized space.
baseline_val = (0.0 - 0.286) / 0.353         # zero pixel after our normalisation

# Pick one image of each of the first 6 classes
images, labels = [], []
seen = set()
for x, y in test:
    y = int(y)
    if y in seen or y >= 6: continue
    images.append(x); labels.append(y); seen.add(y)
    if len(seen) == 6: break

batch = torch.stack(images).to(DEVICE)
attrs = ig.attribute(batch, target=torch.tensor(labels).to(DEVICE),
                     baselines=torch.full_like(batch, baseline_val),
                     n_steps=50)
attrs = attrs.detach().cpu().numpy()

# Visualize
fig, axes = plt.subplots(2, 6, figsize=(13, 4.5))
for i in range(6):
    img = batch[i, 0].detach().cpu().numpy()
    a = attrs[i, 0]
    axes[0, i].imshow(img, cmap="gray")
    axes[0, i].set_title(classes[labels[i]], fontsize=9)
    axes[0, i].axis("off")
    vmax = max(abs(a.min()), abs(a.max()))
    axes[1, i].imshow(a, cmap="coolwarm", vmin=-vmax, vmax=vmax)
    axes[1, i].axis("off")
axes[0, 0].set_ylabel("input"); axes[1, 0].set_ylabel("IG")
fig.suptitle("Integrated Gradients on a small CNN — black-pixel baseline, n_steps=50",
             fontsize=11)
plt.tight_layout(); plt.show()


**Reading.** Red regions argue for the predicted class; blue regions argue against
it. For most clothing classes the attribution concentrates on the silhouette boundary
— the network learns the *shape*, not the texture, which is sensible for this dataset.

### Baseline sensitivity

A common worry: the attribution depends on the baseline. Let's verify on one image.

In [ ]:
x0 = batch[:1]
y0 = torch.tensor([labels[0]]).to(DEVICE)

# Three baselines: black, mean-of-training, and gaussian noise
bg_black = torch.full_like(x0, baseline_val)
bg_mean  = torch.zeros_like(x0)   # mean is exactly zero after our normalisation
bg_noise = torch.randn_like(x0) * 0.3 + baseline_val

fig, axes = plt.subplots(1, 4, figsize=(11, 3))
axes[0].imshow(x0[0, 0].detach().cpu().numpy(), cmap="gray")
axes[0].set_title("input"); axes[0].axis("off")
for ax, b, name in zip(axes[1:], [bg_black, bg_mean, bg_noise],
                       ["black", "mean", "noise"]):
    a = ig.attribute(x0, target=y0, baselines=b, n_steps=50)[0, 0].detach().cpu().numpy()
    vmax = max(abs(a.min()), abs(a.max()))
    ax.imshow(a, cmap="coolwarm", vmin=-vmax, vmax=vmax)
    ax.set_title(f"baseline = {name}"); ax.axis("off")
plt.tight_layout(); plt.show()


**Reading.** All three baselines give qualitatively similar attributions — the
silhouette dominates — but the precise pattern is different. **If your paper claim
hangs on a specific attribution pixel, you must justify the baseline.** When in doubt,
report the result with at least two baselines and note the agreement / disagreement.

## What to do differently in your own research

- For **tree models**, the default is **SHAP TreeExplainer**. No baseline question,
  exact attribution, satisfying axioms. Just use it.
- For **text models**, **LIME** is fast and intuitive but unstable. If you need a single
  defensible figure for a paper, run LIME with `num_samples >= 5000` and average over
  several random seeds, or switch to **SHAP** with token-presence features.
- For **deep models on continuous inputs**, **Captum IG** is the default. Always
  report the **baseline** and run it with at least two baselines as a sensitivity check.
- **Sanity-check your method**: run it on a randomly-initialized copy of the model. If
  the attribution looks the same, the method is not explaining the model.

### Next

Exercises in `exercises/`. Try them, then compare to the reference solutions.
